In [ ]:
dbutils.widgets.text("ticker", "AAPL")
dbutils.widgets.text("mic", "XNAS")
dbutils.widgets.text("trade_date", "2024-02-07")

ticker = dbutils.widgets.get("ticker")
mic = dbutils.widgets.get("mic")
trade_date = dbutils.widgets.get("trade_date")

In [ ]:
# ── Table configuration ─────────────────────────────────────────────────
# Set USE_SAMPLE_DATA=true/false as an environment variable (or Databricks
# cluster/job env var) to switch between trial views (DD-1167) and full
# production views without changing any code.
# Override catalog names via MARKET_DATA_CATALOG / DERIVED_DATA_CATALOG.
import os as _os

_s = _os.environ.get("USE_SAMPLE_DATA", "").lower()
USE_SAMPLE_DATA = True if _s == "true" else (False if _s == "false" else False)

MARKET_DATA_CATALOG  = _os.environ.get("MARKET_DATA_CATALOG",  "market_data")
DERIVED_DATA_CATALOG = _os.environ.get("DERIVED_DATA_CATALOG", "derived_data")

if USE_SAMPLE_DATA:
    # DD-1167 trial views — 6-month rolling window, no subscription required
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.REF_GLOBAL_PROD"
    ANALYTICS_TBL               = f"{MARKET_DATA_CATALOG}.ANA_GLOBAL_PROD"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.TRDS_GLOBAL_PROD"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.TDSP_GLOBAL_PROD"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.L2_GLOBAL_PROD"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.CBBO_GLOBAL_PROD"
    DAILY_CLASSIFIED_TRADES_TBL = f"{MARKET_DATA_CATALOG}.DCT_GLOBAL_PROD"
else:
    # Full production views
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_reference"
    ANALYTICS_TBL               = f"{DERIVED_DATA_CATALOG}.vw_analytics"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.vw_trades"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.vw_trades_plus"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.vw_l2_data_equity"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.vw_cbbo_equity"
    DAILY_CLASSIFIED_TRADES_TBL = f"{DERIVED_DATA_CATALOG}.vw_daily_classified_trades"

## BMLL Notebook Educational Series
### Multi-Level Order Book Imbalance and Short-Horizon Price Impact

This analysis evaluates multi-level order book imbalance as a real-time indicator of liquidity pressure and short-term price movement.

Using 5 levels of depth from BMLL order book data, we construct a normalized imbalance measure that captures aggregate queue pressure across the visible book.

We then examine its relationship with short-horizon mid-price returns to assess predictive content and potential application in execution timing and liquidity regime detection.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import functions as F

In [ ]:
l2 = (
    spark.table("prod_lakehouse.market_data.l2_data_equity")
    .filter(F.col("TradeDate") == "2025-04-03")
    .filter(F.col("Ticker") == "AAPL")
    .filter(F.col("MIC") == "XNAS")
)

In [ ]:
decay = 0.5
weights = [float(np.exp(-decay * i)) for i in range(5)]

l2 = l2.withColumn(
    "weighted_bid",
    sum(F.col(f"BidQuantity{i+1}") * weights[i] for i in range(5))
).withColumn(
    "weighted_ask",
    sum(F.col(f"AskQuantity{i+1}") * weights[i] for i in range(5))
).withColumn(
    "imbalance_5lvl",
    F.try_divide(
        F.col("weighted_bid") - F.col("weighted_ask"),
        F.col("weighted_bid") + F.col("weighted_ask")
    )
)


l2 = l2.withColumn(
    "mid_price",
    0.5 * (F.col("BidPrice1") + F.col("AskPrice1"))
)


l2_1min = (
    l2
    .groupBy(F.window("EventTimestamp", "1 minute"))
    .agg(
        F.avg("mid_price").alias("mid_price"),
        F.avg("imbalance_5lvl").alias("imbalance_5lvl")
    )
    .select(
        F.col("window.start").alias("event_timestamp"),
        "mid_price",
        "imbalance_5lvl"
    )
    .orderBy("event_timestamp")
)

In [ ]:
df = l2_1min.toPandas().sort_values("event_timestamp")

df["fwd_return"] = df["mid_price"].pct_change().shift(-1)
df["imb_decile"] = pd.qcut(df["imbalance_5lvl"], 10, labels=False)

##
### Plot 1: Imbalance vs Midprice (Time Series)
The 5-level imbalance captures aggregate queue pressure across the visible book. Sustained positive imbalance typically coincides with upward price drift, while sustained negative imbalance aligns with short-term price weakness. <br> Periods of divergence may indicate temporary liquidity absorption or regime shifts in market resilience.

In [ ]:
df = df.sort_values("event_timestamp").drop_duplicates("event_timestamp")

df["imbalance_5lvl"] = df["imbalance_5lvl"].fillna(0)

fig, ax1 = plt.subplots(figsize=(12,6))

ax1.plot(df["event_timestamp"], df["mid_price"], color="black")
ax1.set_ylabel("Mid Price")

ax2 = ax1.twinx()
ax2.plot(df["event_timestamp"], df["imbalance_5lvl"],
         color="blue", alpha=0.6)
ax2.set_ylabel("Imbalance")

plt.title("5-Level Imbalance vs Mid Price")
plt.tight_layout()
plt.show()

##
### Plot 2: Distribution of Imbalance
The imbalance distribution is centered near zero but exhibits meaningful tail observations, indicating that significant liquidity pressure regimes occur with non-trivial frequency. <br> This suggests the signal has sufficient variation to potentially explain short-horizon price dynamics.

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df["imbalance_5lvl"], bins=50, kde=True)
plt.title("Distribution of 5-Level Imbalance")
plt.show()

##
### Plot 3: Conditional Returns by Imbalance Decile
If forward returns increase monotonically with imbalance deciles, this would indicate that extreme positive queue pressure is associated with short-horizon price continuation.

In [ ]:
df["fwd_return"] = df["mid_price"].pct_change().shift(-1)
df["imb_decile"] = pd.qcut(df["imbalance_5lvl"], 10, labels=False)

decile_returns = df.groupby("imb_decile")["fwd_return"].mean()

plt.figure(figsize=(10,5))
decile_returns.plot(kind="bar")
plt.title("Average Forward Return by Imbalance Decile")
plt.show()

##
## Trading Strategy Return Based on Imbalance

We construct a directional intraday trading strategy based on 5-level order book imbalance aggregated to 1-minute intervals.

A long position is initiated when normalized imbalance exceeds +0.10, indicating sustained buy-side queue pressure across the top five levels of the order book.

A short position is initiated when imbalance falls below -0.10, indicating sell-side pressure.

Positions are held until a contrary signal is observed, and returns are calculated using next-period mid-price changes.

Strategy performance is evaluated using cumulative returns, Sharpe ratio, and maximum drawdown.

In [ ]:
df_agg = df.set_index("event_timestamp").resample("1T").mean().reset_index()

df_agg = df_agg.sort_values("event_timestamp")

df_agg["forward_return"] = df_agg["mid_price"].pct_change().shift(-1)

df_agg["position"] = 0
df_agg.loc[df_agg["imbalance_5lvl"] > 0.1, "position"] = 1
df_agg.loc[df_agg["imbalance_5lvl"] < -0.1, "position"] = -1

df_agg["position"] = df_agg["position"].ffill().fillna(0)

# IMPORTANT: multiply by forward return
df_agg["strategy_return"] = df_agg["position"] * df_agg["forward_return"]

# Remove final NaN row
df_agg = df_agg.dropna(subset=["strategy_return"])

df_agg["cum_pnl"] = (1 + df_agg["strategy_return"]).cumprod()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(df_agg["event_timestamp"],
         df_agg["cum_pnl"],
         label="Strategy Cumulative PnL")

plt.title("Backtest: Imbalance-Based Strategy")
plt.xlabel("Time")
plt.ylabel("Cumulative PnL")
plt.legend()
plt.tight_layout()
plt.show()